# 💻 Nærmeste nabo-analyse

Når vi analyserer romlige data, er det ofte ikke bare interessant å vite *om* to geometrier overlapper eller ligger innenfor hverandre (som vi så med `sjoin()` og `overlay()`), men også **hvor nærme** de er hverandre, selv om de ikke overlapper. Dette kalles en *nærmeste nabo-analyse* (engelsk: *nearest neighbour analysis*), og brukes mye i praktisk GIS-arbeid, for eksempel for å finne:

- den nærmeste bussholdeplassen fra en adresse
- det nærmeste sykehuset fra et ulykkessted
- hvilke boliger som ligger lengst fra en tjeneste (tilgjengelighetsanalyse)

I denne notebooken (inspirert av [kapittel 6.7 i Python for Geographic Data Analysis](https://python-gis-book.readthedocs.io/en/latest/part2/chapter-06/nb/07-nearest-neighbour.html)) skal vi finne det **nærmeste kjøpesenteret** for hver adresse i adressedatasettet vi har brukt tidligere i kurset.


In [ ]:
import pathlib 
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_MAPPE = NOTEBOOK_PATH / "data"

## Last inn inngangsdata

Vi bruker to datasett:

1. `adresser`, adressepunktene fra Oslo som vi har brukt i tidligere forelesninger
2. `kjopesentre`, buffere rundt kjøpesentre i Oslo som vi har brukt tidligere

In [ ]:
import geopandas

adresser = geopandas.read_file(DATA_MAPPE / "oslo_adresser" / "adresser.geojson")
kjopesentre = geopandas.read_file(DATA_MAPPE / "oslo_adresser" / "shopping_centres.geojson")

kjopesentre.head()

Som vi ser er `kjopesentre` polygoner, med kolonnene `navn` og `adr` i tillegg til `geometry`. `sjoin_nearest()`, som vi skal bruke senere i denne notebooken, fungerer uavhengig av hvilken geometritype vi har (punkt, linje eller polygon) — men polygonene her er egentlig buffere vi har lagd tidligere for hvert kjøpesenter. For denne analysen er vi egentlig bare interessert i *hvor* hvert kjøpesenter ligger, ikke i bufferen. La oss derfor, som vi gjorde med `.centroid` i en tidligere forelesning, representere hvert kjøpesenter med et enkelt punkt i stedet for en polygon:

In [ ]:
kjopesentre["geometry"] = ____
kjopesentre.head()

## Sjekke og tilpasse koordinatsystem

Siden vi skal regne ut faktiske avstander, må dataene våre være i et projisert koordinatsystem med en metrisk enhet (meter), og begge lagene må dele samme CRS. La oss sjekke:

In [ ]:
print(adresser.crs)
print(kjopesentre.crs)

`adresser` er i `EPSG:4326` (lengde-/breddegrader), mens `kjopesentre` allerede er i `EPSG:25832`, et projisert koordinatsystem med meter som enhet som passer godt for norske data. La oss reprojisere `adresser` til samme CRS:

In [ ]:
adresser = ____
assert adresser.crs == kjopesentre.crs, "CRS er ikke identiske"

## Konseptet: `shapely.ops.nearest_points()`

Før vi bruker det praktiske verktøyet i geopandas, er det nyttig å se selve ideen bak en nærmeste nabo-analyse på to enkeltgeometrier med `shapely`. Funksjonen [`nearest_points()`](https://shapely.readthedocs.io/en/stable/manual.html#nearest-points) tar inn to geometrier, og returnerer et par med de to punktene (ett fra hver geometri) som ligger nærmest hverandre.

La oss prøve dette med den første adressen i datasettet, mot *alle* kjøpesentrene samlet som én geometri (ved hjelp av `union_all()`, som vi har brukt tidligere):

In [ ]:
import shapely.ops

en_adresse = ____
#naermeste_punkter = ____
#naermeste_punkter

`nearest_points()` gir oss tilbake selve punktene, men ikke noe mer informasjon: vi får ikke vite *hvilket* kjøpesenter (hvilken rad) som var nærmest, og heller ikke *hvor langt* unna det er. For å finne det, må vi regne ut avstanden selv med `.distance()`, og vi måtte i tillegg iterere over alle radene i `adresser` for å gjøre dette for hver adresse enkeltvis — det ville fungert, men ville vært tungvint og ville tatt mye lenger tid å kjøre for et stort datasett.

Derfor har geopandas et eget verktøy som gjør akkurat denne jobben for oss, mye raskere og med én kommando.

## Nærmeste nabo-kobling med `sjoin_nearest()`

Metoden [`sjoin_nearest()`](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin_nearest.html) fungerer på samme måte som `sjoin()` som vi brukte i en tidligere forelesning, men i stedet for å koble sammen rader basert på et romlig predikat (som `within`), kobler den hver rad i venstre GeoDataFrame til den *nærmeste* raden i høyre GeoDataFrame — uavhengig av om de overlapper eller ikke.

Med parameteren `distance_col` kan vi også få avstanden til det nærmeste objektet lagt til som en egen kolonne:

In [ ]:
adresser_med_naermeste = ____
adresser_med_naermeste.head()

Som med `sjoin()` har vi nå attributtene fra begge lagene i resultatet, og i tillegg en kolonne `avstand_meter` som viser avstanden (i meter, siden CRS-en vår bruker meter som enhet) til det nærmeste kjøpesenteret for hver adresse.

La oss finne adressen som ligger *lengst* unna sitt nærmeste kjøpesenter:

In [ ]:
adresser_med_naermeste.sort_values("avstand_meter", ascending=False).head()

:::{admonition} Sjekk din forståelse
:class: hint

Se på adressen med lengst `avstand_meter` ovenfor. Hvilken bydel tror du den ligger i, og gir det mening at den ligger langt fra et kjøpesenter? Du kan sjekke dette selv ved å gjøre en romlig kobling (`sjoin()`) mellom denne adressen og bydelslaget fra en tidligere forelesning.
:::

## Visualisere resultatet

La oss plotte adressene, farget etter hvor langt de er fra sitt nærmeste kjøpesenter, sammen med kjøpesentrene selv:

In [ ]:
ax = adresser_med_naermeste.plot(
    figsize=(10, 10),
    column="avstand_meter",
    cmap="viridis",
    markersize=20,
    legend=True
)
kjopesentre.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)
ax.set_title("Avstand (meter) til nærmeste kjøpesenter")

## Avgrense søket med `max_distance`

Noen ganger er vi bare interessert i objekter som faktisk har *noe* innen en gitt avstand, ikke bare det nærmeste uansett hvor langt unna det er. `sjoin_nearest()` har derfor en `max_distance`-parameter: hvis ingen rad i høyre GeoDataFrame er innenfor denne avstanden, får raden `NaN`-verdier i stedet (så lenge vi bruker `how="left"`, som beholder alle adressene uavhengig av treff).

La oss finne ut hvor mange adresser som *ikke* har et kjøpesenter innen 1000 meter:

In [ ]:
adresser_naer_senter = ____

antall_uten_senter_i_naerheten = adresser_naer_senter["avstand_meter"].isna().sum()
antall_uten_senter_i_naerheten

Det er mange flere bruksområder for nærmeste nabo-analyser i [geopandas sin egen dokumentasjon](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.sjoin_nearest.html), og i [Python for Geographic Data Analysis, kapittel 6.7](https://python-gis-book.readthedocs.io/en/latest/part2/chapter-06/nb/07-nearest-neighbour.html), som blant annet også viser hvordan man kan finne *flere* nærmeste naboer (ikke bare den aller nærmeste) ved hjelp av biblioteket `scikit-learn`.